# LoanGuard AI - MLOps Lab
## Data & Model Versioning + Registry

---

### Your Mission

You've just joined **LoanGuard AI**, a fintech startup that predicts loan defaults.
Their ML team has a serious problem: models keep breaking in production and nobody can explain why.
Predictions changed last week, accuracy dropped, and a roll back took 3 days.

**Your job:** Fix their MLOps stack - layer by layer - using data versioning, experiment tracking, model versioning, and a model registry.

---

### Lab Structure

| Task | Topic | Real-World Problem You Solve |
|------|-------|------------------------------|
| **Task 1** | Data Versioning with DVC | Reproduce a model that broke 2 months ago |
| **Task 2** | Experiment Tracking with MLflow | Compare 3 model candidates before deploying |
| **Task 3** | Model Versioning | Name and register a model so the team can roll back |
| **Task 4** | Registry Lifecycle | Promote a model through staging → production → archived |

---

### How the Lab Works

- Each task has a ** Beginner path** (fill in the blanks) and a ** Advanced path** (write from scratch)
- Use the ** Hint system** - try first, then reveal hints one at a time
- Run the ** Auto-check** cells to verify your work before moving on
- **You do not need to run DVC or MLflow CLI** - everything is done in Python

---

### Setup - Run this first

In [76]:
# installing required library for experiment tracking
!pip install mlflow

85676.13s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [3]:
# --- SETUP - Run this cell before anything else ----------------------
import os, json, hashlib, shutil, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pickle, time

warnings.filterwarnings("ignore")

# Lab workspace
LAB_DIR = Path("loanGuard_lab")
LAB_DIR.mkdir(exist_ok=True)
(LAB_DIR / "data").mkdir(exist_ok=True)
(LAB_DIR / "models").mkdir(exist_ok=True)
(LAB_DIR / "dvc_cache").mkdir(exist_ok=True)
MLFLOW_URI = f"sqlite:///{LAB_DIR}/mlflow.db"

print(" Setup complete - LoanGuard AI Lab is ready")
print(f"   Workspace : {LAB_DIR.resolve()}")
print(f"   MLflow DB : {MLFLOW_URI}")

 Setup complete - LoanGuard AI Lab is ready
   Workspace : /home/ciuser/allfiles/students/HemaSree_Jupyter/MLOPS/loanGuard_lab
   MLflow DB : sqlite:///loanGuard_lab/mlflow.db


### Simulated Data Generator

LoanGuard AI receives loan applications with customer financial data.
Run the cell below to simulate **three versions** of their dataset - representing how data changed over time.

>  **Don't modify this cell.** It simulates real-world upstream data changes you'll need to detect and version.

In [43]:
# ----- DATA SIMULATOR - Do not modify ----------------------------------
np.random.seed(42)

def make_loan_dataset(n=2000, schema_version="v1", seed=42):
    """Simulate LoanGuard's loan application data at different points in time."""
    rng = np.random.RandomState(seed)
    age            = rng.randint(22, 65, n)
    income         = rng.normal(55000, 20000, n).clip(15000, 200000)
    loan_amount    = rng.normal(12000, 5000, n).clip(1000, 50000)
    credit_score   = rng.normal(680, 80, n).clip(300, 850)
    employment_yrs = rng.randint(0, 30, n)
    debt_ratio     = rng.uniform(0.05, 0.65, n)

    # Default probability driven by risk factors
    risk = (
        -0.3 * (credit_score - 680) / 80
        + 0.4 * debt_ratio
        + 0.2 * (loan_amount / income)
        - 0.1 * employment_yrs / 10
    )
    default = (1 / (1 + np.exp(-risk * 3 + rng.normal(0, 0.5, n))) > 0.5).astype(int)

    df = pd.DataFrame({
        "age": age, "annual_income": income.round(2),
        "loan_amount": loan_amount.round(2), "credit_score": credit_score.round(1),
        "employment_years": employment_yrs, "debt_to_income": debt_ratio.round(4),
        "default": default
    })

    # v2: upstream adds a new column (common real-world schema drift)
    if schema_version in ("v2", "v3"):
        df["num_accounts"] = rng.randint(1, 10, n)

    # v3: a data quality issue introduces nulls (another common real-world problem)
    if schema_version == "v3":
        null_idx = rng.choice(n, size=int(n * 0.15), replace=False)
        df.loc[null_idx, "credit_score"] = np.nan

    return df

# Generate and save three dataset versions
datasets = {}
for ver, schema, seed in [("v1", "v1", 42), ("v2", "v2", 42), ("v3", "v3", 99)]:
    df = make_loan_dataset(schema_version=schema, seed=seed)
    path = LAB_DIR / "data" / f"loans_{ver}.csv"
    df.to_csv(path, index=False)
    datasets[ver] = df
    print(f"  loans_{ver}.csv  |  {len(df):,} rows  |  {df.shape[1]} cols  |  "
          f"{df['default'].mean():.1%} default rate  |  nulls: {df.isnull().sum().sum()}")

print("\n Three dataset versions ready")

  loans_v1.csv  |  2,000 rows  |  7 cols  |  58.1% default rate  |  nulls: 0
  loans_v2.csv  |  2,000 rows  |  8 cols  |  58.1% default rate  |  nulls: 0
  loans_v3.csv  |  2,000 rows  |  8 cols  |  54.6% default rate  |  nulls: 300

 Three dataset versions ready


---
# TASK 1 - Data Versioning with DVC
> Can you prove which data trained the broken model?
---

##  Task 1 - Data Versioning with DVC

### Scenario
> LoanGuard's model accuracy dropped from 87% to 71% last month. The engineering team says 'the code didn't change'. You suspect a silent data change. You need to (a) version the dataset snapshots so they can be reproduced later, (b) detect what changed between versions, and (c) implement schema validation to catch future changes automatically.

### Objective
By the end of this task you will be able to:
- Simulate DVC-style versioning using checksums and metadata (`.dvc` pointer files)
- Compare two dataset versions and identify schema/distribution drift
- Write a schema validator that raises an alert before a bad dataset enters the pipeline

---

###  Background - How DVC Versioning Works

DVC doesn't store data in Git. It stores a **tiny pointer file** (`.dvc`) containing:
- A **checksum** (MD5 hash) of the data file
- The file path and size
- The remote storage location

When you run `dvc pull`, DVC uses the checksum to fetch the exact data file.

In this task you'll **simulate this mechanism in pure Python** - the same logic DVC uses internally.

---

In [5]:
#  BEGINNER 1A - Create DVC-style pointer files
# ----------------------------------------------------------------------------------------

def create_dvc_pointer(csv_path):
    """
    Create a DVC-style metadata pointer for a dataset file.
    """
    path = Path(csv_path)

    # Step 1: Compute MD5 checksum of the file
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    # Step 2: Load the CSV to get row/column counts
    df = pd.read_csv(path)

    # Step 3: Build and return the metadata dictionary
    return {
        "md5": md5,
        "path": str(path),
        "size": os.path.getsize(path),   # file size in bytes
        "rows": len(df),                 # number of rows
        "cols": df.shape[1],             # number of columns
        "created_at": datetime.now().isoformat()
    }

# ----- Test it -----
pointer_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")
print("DVC pointer for loans_v1:")
print(json.dumps(pointer_v1, indent=2))

DVC pointer for loans_v1:
{
  "md5": "32749950b434fb1f5e15bc9573a4ccd0",
  "path": "loanGuard_lab/data/loans_v1.csv",
  "size": 76059,
  "rows": 2000,
  "cols": 7,
  "created_at": "2026-03-12T13:55:00.137829"
}


###  1B - Version all three datasets and save pointer files

Call your function on all three loan datasets and save each pointer as a `.dvc` JSON file.

In [6]:
#  BEGINNER 1B - Version all three datasets
# --------------------------------------------------

dvc_store = {}   # will hold {version: pointer_dict}

for version in ["v1", "v2", "v3"]:
    csv_path = LAB_DIR / "data" / f"loans_{version}.csv"

    # Step 1: Create pointer using your function above
    pointer = create_dvc_pointer(csv_path)

    # Step 2: Save pointer to a .dvc JSON file
    dvc_file = LAB_DIR / "data" / f"loans_{version}.csv.dvc"
    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    dvc_store[version] = pointer
    print(f"  loans_{version}.csv  →  md5: {pointer['md5'][:12]}...  "
          f"({pointer['rows']} rows, {pointer['cols']} cols)")

print("\n All versions tracked")

  loans_v1.csv  →  md5: 32749950b434...  (2000 rows, 7 cols)
  loans_v2.csv  →  md5: 3b43198058ae...  (2000 rows, 8 cols)
  loans_v3.csv  →  md5: b8ac0fe8ebdd...  (2000 rows, 8 cols)

 All versions tracked


###  1C - Compare two versions and detect drift

Write a function that compares two DVC pointers and the underlying CSVs to detect:
- Schema changes (new/removed columns)
- Row count changes
- Statistical drift in numeric columns (mean shift > 5%)
- Data quality issues (null counts)

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Load both CSVs with `pd.read_csv()`. Compare `df1.columns` and `df2.columns` using set operations to find added/removed columns.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For numeric drift, iterate over shared columns: `abs(df2[col].mean() - df1[col].mean()) / (df1[col].mean() + 1e-9)`. Flag anything above 0.05.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def compare_versions(ptr_a, ptr_b):
    df_a, df_b = pd.read_csv(ptr_a['path']), pd.read_csv(ptr_b['path'])
    cols_a, cols_b = set(df_a.columns), set(df_b.columns)
    report = {
        'added_cols':   list(cols_b - cols_a),
        'removed_cols': list(cols_a - cols_b),
        'row_delta':    ptr_b['rows'] - ptr_a['rows'],
        'null_delta':   int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        'drift': {}
    }
    for col in cols_a & cols_b:
        if df_a[col].dtype in [float, int]:
            shift = abs(df_b[col].mean() - df_a[col].mean()) / (abs(df_a[col].mean()) + 1e-9)
            if shift > 0.05:
                report['drift'][col] = round(shift, 4)
    return report
```

</details>

In [9]:
#  BEGINNER 1C - Detect drift between dataset versions
# ----------------------------------------------------------

def compare_dataset_versions(ptr_a, ptr_b):
    """
    Compare two versioned datasets and produce a drift report.

    Returns a dictionary containing:
      - added_cols    : columns present in dataset B but not in dataset A
      - removed_cols  : columns present in dataset A but not in dataset B
      - row_delta     : change in row count between datasets
      - null_delta    : change in total null values
      - drift         : dictionary showing numeric column drift
    """

    # ------------------------------------------------------------------
    #  Step 1: Load both datasets
    # ------------------------------------------------------------------
    # We read the dataset paths stored inside the DVC pointer files
    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    # ------------------------------------------------------------------
    #  Step 2: Convert column lists to sets
    # ------------------------------------------------------------------
    # Sets allow us to easily compare which columns were added or removed
    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    # ------------------------------------------------------------------
    #  Step 3: Create the drift report dictionary
    # ------------------------------------------------------------------
    report = {

        # Columns that appear in dataset B but not in dataset A
        "added_cols": list(cols_b - cols_a),

        # Columns that existed in dataset A but were removed in dataset B
        "removed_cols": list(cols_a - cols_b),

        # Change in number of rows
        # Example: if A has 1000 rows and B has 1100 → row_delta = 100
        "row_delta": ptr_b["rows"] - ptr_a["rows"],

        # Difference in total number of null values
        # We count all missing values in each dataset and subtract them
        "null_delta": int(
            df_b.isnull().sum().sum() - df_a.isnull().sum().sum()
        ),

        # Dictionary that will store columns showing statistical drift
        "drift": {}
    }

    # ------------------------------------------------------------------
    #  Step 4: Identify numeric columns shared by both datasets
    # ------------------------------------------------------------------
    shared_numeric = [
        c for c in cols_a & cols_b
        if df_a[c].dtype in [np.float64, np.int64]
    ]

    # ------------------------------------------------------------------
    #  Step 5: Detect statistical drift
    # ------------------------------------------------------------------
    for col in shared_numeric:

        # Calculate mean value of the column in both datasets
        mean_a = df_a[col].mean()
        mean_b = df_b[col].mean()

        # Compute relative mean shift
        # Adding a very small value (1e-9) prevents division by zero
        shift = abs(mean_b - mean_a) / (abs(mean_a) + 1e-9)

        # If the change is greater than 5%, mark it as drift
        if shift > 0.05:
            report["drift"][col] = round(shift, 4)

    # Return the complete drift report
    return report


# ----------------------------------------------------------------------
#  Compare dataset versions
# ----------------------------------------------------------------------

print("═" * 55)
print("Comparing loans_v1  →  loans_v2")
print("═" * 55)

# Compare version 1 with version 2
r12 = compare_dataset_versions(dvc_store["v1"], dvc_store["v2"])

# Print drift report
print(json.dumps(r12, indent=2))


print("\n" + "═" * 55)
print("Comparing loans_v2  →  loans_v3  (the bad version!)")
print("═" * 55)

# Compare version 2 with version 3
r23 = compare_dataset_versions(dvc_store["v2"], dvc_store["v3"])

# Print drift report
print(json.dumps(r23, indent=2))

═══════════════════════════════════════════════════════
Comparing loans_v1  →  loans_v2
═══════════════════════════════════════════════════════
{
  "added_cols": [
    "num_accounts"
  ],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 0,
  "drift": {}
}

═══════════════════════════════════════════════════════
Comparing loans_v2  →  loans_v3  (the bad version!)
═══════════════════════════════════════════════════════
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "default": 0.0594
  }
}


###  1D - Schema Validator

Write a schema validator that checks incoming data against a **golden schema** (captured from a known-good dataset).
It must raise a `ValueError` with a descriptive message if validation fails.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Store the golden schema as a dict: `{'columns': list, 'dtypes': dict, 'null_thresholds': dict}`. Compare incoming data against it.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Check three things in order: (1) all expected columns present, (2) null rate per column below threshold, (3) value ranges reasonable.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def build_golden_schema(df, null_tolerance=0.05):
    return {
        'columns': list(df.columns),
        'null_rates': {c: df[c].isnull().mean() for c in df.columns},
        'null_tolerance': null_tolerance,
        'ranges': {c: (df[c].min(), df[c].max())
                   for c in df.select_dtypes(include=np.number).columns}
    }

def validate_schema(df, schema):
    errors = []
    missing = set(schema['columns']) - set(df.columns)
    if missing: errors.append(f'Missing columns: {missing}')
    for col in schema['columns']:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema['null_tolerance']:
                errors.append(f'{col}: null rate {null_rate:.1%} > tolerance')
    if errors:
        raise ValueError('Schema validation failed:\n' + '\n'.join(errors))
    return True
```

</details>

In [10]:
#  BEGINNER 1D - Schema Validator
# ----------------------------------------------------------

def build_golden_schema(df, null_tolerance=0.05):
    """
    Capture the schema of a known-good dataset.

    Parameters
    ----------
    df : DataFrame
        The trusted dataset used as reference (golden dataset)

    null_tolerance : float
        Maximum allowed null percentage change when validating

    Returns
    -------
    dict containing:
        columns         → list of dataset columns
        null_rates      → null percentage per column
        null_tolerance  → allowed null increase
        ranges          → min/max values for numeric columns
    """

    schema = {}

    # ------------------------------------------------------
    # Store column names
    # ------------------------------------------------------
    schema["columns"] = list(df.columns)

    # ------------------------------------------------------
    # Calculate null rate per column
    # null rate = number_of_nulls / total_rows
    # ------------------------------------------------------
    schema["null_rates"] = {
        col: df[col].isnull().mean() for col in df.columns
    }

    # ------------------------------------------------------
    # Store allowed tolerance for null increase
    # ------------------------------------------------------
    schema["null_tolerance"] = null_tolerance

    # ------------------------------------------------------
    # Capture numeric ranges (min and max values)
    # This helps detect abnormal values later
    # ------------------------------------------------------
    ranges = {}

    for col in df.select_dtypes(include=["float64", "int64"]).columns:
        ranges[col] = {
            "min": df[col].min(),
            "max": df[col].max()
        }

    schema["ranges"] = ranges

    return schema


def validate_schema(df, schema):
    """
    Validate a new dataset against the golden schema.

    If validation fails, raise ValueError listing all issues.
    """

    errors = []

    # ------------------------------------------------------
    # Check 1: All expected columns are present
    # ------------------------------------------------------
    expected_cols = set(schema["columns"])
    actual_cols = set(df.columns)

    missing_cols = expected_cols - actual_cols
    extra_cols = actual_cols - expected_cols

    if missing_cols:
        errors.append(f"Missing columns: {list(missing_cols)}")

    if extra_cols:
        errors.append(f"Unexpected columns: {list(extra_cols)}")

    # ------------------------------------------------------
    # Check 2: Null rate per column is within tolerance
    # ------------------------------------------------------
    for col in expected_cols & actual_cols:

        current_null_rate = df[col].isnull().mean()
        golden_null_rate = schema["null_rates"].get(col, 0)

        allowed = golden_null_rate + schema["null_tolerance"]

        if current_null_rate > allowed:
            errors.append(
                f"High null rate in '{col}': "
                f"{current_null_rate:.2%} > allowed {allowed:.2%}"
            )

    # ------------------------------------------------------
    # If any validation errors exist → raise exception
    # ------------------------------------------------------
    if errors:
        raise ValueError(
            "Schema validation FAILED:\n"
            + "\n".join(f"   {e}" for e in errors)
        )

    print(" Schema validation passed")
    return True


# ----------------------------------------------------------
# Test: Use v1 as the golden dataset
# ----------------------------------------------------------

golden = build_golden_schema(datasets["v1"])

print("Golden schema columns:", golden.get("columns", []))
print()

# ----------------------------------------------------------
# Validate dataset version v2
# ----------------------------------------------------------
print("Validating v2 (has extra column)...")

try:
    validate_schema(datasets["v2"], golden)

except ValueError as e:
    print(f"Caught validation error:\n{e}")

print()

# ----------------------------------------------------------
# Validate dataset version v3
# ----------------------------------------------------------
print("Validating v3 (has nulls + extra column)...")

try:
    validate_schema(datasets["v3"], golden)

except ValueError as e:
    print(f"Caught validation error:\n{e}")

Golden schema columns: ['age', 'annual_income', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'default']

Validating v2 (has extra column)...
Caught validation error:
Schema validation FAILED:
   Unexpected columns: ['num_accounts']

Validating v3 (has nulls + extra column)...
Caught validation error:
Schema validation FAILED:
   Unexpected columns: ['num_accounts']
   High null rate in 'credit_score': 15.00% > allowed 5.00%


In [11]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert pointer_v1.get('md5') and len(pointer_v1['md5']) == 32, 'md5 must be a 32-char hex string'
    passed.append('DVC pointer created')
except Exception as e:
    failed.append(('DVC pointer created', str(e)))

try:
    assert all(k in pointer_v1 for k in ['md5','path','size','rows','cols']), 'Missing keys in pointer'
    passed.append('DVC pointer has required keys')
except Exception as e:
    failed.append(('DVC pointer has required keys', str(e)))

try:
    assert len(dvc_store) == 3, 'dvc_store should have v1, v2, v3'
    passed.append('All 3 versions stored')
except Exception as e:
    failed.append(('All 3 versions stored', str(e)))

try:
    assert r23.get('null_delta', 0) > 0, 'Should detect null increase in v3'
    passed.append('Drift detected v2→v3')
except Exception as e:
    failed.append(('Drift detected v2→v3', str(e)))

try:
    assert True, 'Manually verify the validator raised an error for v3'
    passed.append('Schema validator catches v3')
except Exception as e:
    failed.append(('Schema validator catches v3', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  DVC pointer created
  DVC pointer has required keys
  All 3 versions stored
  Drift detected v2→v3
  Schema validator catches v3

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 1

Implement a complete DVC-style versioning system with the following requirements:

**Requirements:**
1. `create_dvc_pointer(path)` - MD5 checksum, full metadata, saved to `.dvc` file
2. `compare_dataset_versions(ptr_a, ptr_b)` - schema diff, row diff, null diff, per-column statistical drift (mean, std, min, max shifts)
3. `build_golden_schema(df)` + `validate_schema(df, schema)` - catches missing columns, null rate violations, and out-of-range values for numeric columns
4. `reproduce_dataset(version)` - given a version string, loads the correct CSV using the `.dvc` pointer file (simulating `dvc pull`)
5. Demonstrate: show that v3 fails validation, v1 passes, and `reproduce_dataset('v1')` returns the exact same checksum as the v1 pointer

Write all functions in the cell below with no scaffolding.

In [ ]:
#  ADVANCED - Task 1 Full Implementation
# ---------------------------------------
# YOUR CODE HERE

In [16]:
# ==========================================================
# ADVANCED TASK 1 - Mini Dataset Version Control System
# ==========================================================
# This program simulates basic features of a data versioning
# workflow similar to tools like DVC.
#
# Features:
# 1. Create dataset metadata pointer
# 2. Compare dataset versions
# 3. Build a reference schema
# 4. Validate new datasets against the schema
# 5. Reproduce datasets using metadata
# ==========================================================


# ----------------------------------------------------------
# 1. GENERATE DATA POINTER
# ----------------------------------------------------------
def generate_pointer(file_path):
    """
    Create a metadata pointer describing the dataset.
    """

    file_path = Path(file_path)

    # Compute dataset checksum
    hash_obj = hashlib.md5()

    with open(file_path, "rb") as file:
        data = file.read()
        hash_obj.update(data)

    checksum = hash_obj.hexdigest()

    # Load dataset to inspect structure
    df = pd.read_csv(file_path)

    pointer = {
        "checksum": checksum,
        "file": str(file_path),
        "file_size": file_path.stat().st_size,
        "num_rows": df.shape[0],
        "num_cols": df.shape[1],
        "timestamp": datetime.now().isoformat()
    }

    # Save pointer metadata
    pointer_name = file_path.with_suffix(".pointer.json")

    with open(pointer_name, "w") as f:
        json.dump(pointer, f, indent=2)

    return pointer


# ----------------------------------------------------------
# 2. DATASET DIFFERENCE ANALYSIS
# ----------------------------------------------------------
def analyze_versions(meta_a, meta_b):
    """
    Compare two dataset versions and report differences.
    """

    df1 = pd.read_csv(meta_a["file"])
    df2 = pd.read_csv(meta_b["file"])

    columns1 = set(df1.columns)
    columns2 = set(df2.columns)

    results = {}

    # Column differences
    results["new_columns"] = list(columns2 - columns1)
    results["dropped_columns"] = list(columns1 - columns2)

    # Row difference
    results["row_difference"] = meta_b["num_rows"] - meta_a["num_rows"]

    # Null difference
    nulls_1 = df1.isna().sum().sum()
    nulls_2 = df2.isna().sum().sum()

    results["null_change"] = int(nulls_2 - nulls_1)

    # Numeric drift detection
    drift_report = {}

    numeric_cols = df1.select_dtypes(include=np.number).columns

    for column in numeric_cols:

        if column in df2.columns:

            avg1 = df1[column].mean()
            avg2 = df2[column].mean()

            change = abs(avg2 - avg1) / (abs(avg1) + 1e-9)

            if change > 0.05:
                drift_report[column] = round(change, 3)

    results["drift_columns"] = drift_report

    return results


# ----------------------------------------------------------
# 3. CREATE REFERENCE SCHEMA
# ----------------------------------------------------------
def create_reference_schema(df, tolerance=0.05):
    """
    Build a schema from a trusted dataset version.
    """

    schema = {}

    schema["expected_columns"] = list(df.columns)

    schema["null_limits"] = {
        c: df[c].isna().mean()
        for c in df.columns
    }

    schema["allowed_null_tolerance"] = tolerance

    numeric_ranges = {}

    for col in df.select_dtypes(include=np.number).columns:
        numeric_ranges[col] = (df[col].min(), df[col].max())

    schema["value_ranges"] = numeric_ranges

    return schema


# ----------------------------------------------------------
# 4. SCHEMA VALIDATION
# ----------------------------------------------------------
def check_schema(df, schema):
    """
    Ensure dataset follows the reference schema.
    """

    problems = []

    # Missing columns
    expected = set(schema["expected_columns"])
    actual = set(df.columns)

    missing = expected - actual

    if missing:
        problems.append(f"Missing columns -> {missing}")

    # Null validation
    for col in schema["expected_columns"]:

        if col in df.columns:

            current_null_rate = df[col].isna().mean()

            if current_null_rate > schema["allowed_null_tolerance"]:
                problems.append(
                    f"{col} null rate {current_null_rate:.2%} exceeds tolerance"
                )

    # Range validation
    for col, limits in schema["value_ranges"].items():

        if col in df.columns:

            low, high = limits

            if df[col].min() < low or df[col].max() > high:
                problems.append(f"{col} values outside expected range")

    if problems:
        raise ValueError("Schema issues found:\n" + "\n".join(problems))

    print("Dataset passed schema validation")
    return True


# ----------------------------------------------------------
# 5. LOAD DATASET USING POINTER
# ----------------------------------------------------------
def load_dataset(version_name):
    """
    Reproduce dataset using stored metadata.
    """

    pointer_file = LAB_DIR / "data" / f"loans_{version_name}.pointer.json"

    with open(pointer_file) as f:
        meta = json.load(f)

    df = pd.read_csv(meta["file"])

    # Verify checksum
    hash_check = hashlib.md5()

    with open(meta["file"], "rb") as f:
        hash_check.update(f.read())

    if hash_check.hexdigest() != meta["checksum"]:
        raise ValueError("Dataset integrity check failed")

    return df
    # ==========================================================
# TESTING THE SYSTEM
# ==========================================================

print("Creating dataset pointers...\n")

ptr_v1 = generate_pointer(LAB_DIR / "data" / "loans_v1.csv")
ptr_v2 = generate_pointer(LAB_DIR / "data" / "loans_v2.csv")
ptr_v3 = generate_pointer(LAB_DIR / "data" / "loans_v3.csv")

print("Pointers created!\n")


print("Comparing Version v1 -> v2\n")
report1 = analyze_versions(ptr_v1, ptr_v2)
print(json.dumps(report1, indent=2))


print("\nComparing Version v2 -> v3\n")
report2 = analyze_versions(ptr_v2, ptr_v3)
print(json.dumps(report2, indent=2))


print("\nCreating Golden Schema from v1\n")
golden_schema = create_reference_schema(pd.read_csv(ptr_v1["file"]))


print("\nValidating v2 dataset\n")
try:
    check_schema(pd.read_csv(ptr_v2["file"]), golden_schema)
except ValueError as e:
    print(e)


print("\nValidating v3 dataset\n")
try:
    check_schema(pd.read_csv(ptr_v3["file"]), golden_schema)
except ValueError as e:
    print(e)


print("\nReproducing dataset v1 using pointer\n")
df = load_dataset("v1")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Creating dataset pointers...

Pointers created!

Comparing Version v1 -> v2

{
  "new_columns": [
    "num_accounts"
  ],
  "dropped_columns": [],
  "row_difference": 0,
  "null_change": 0,
  "drift_columns": {}
}

Comparing Version v2 -> v3

{
  "new_columns": [],
  "dropped_columns": [],
  "row_difference": 0,
  "null_change": 300,
  "drift_columns": {
    "default": 0.059
  }
}

Creating Golden Schema from v1


Validating v2 dataset

Dataset passed schema validation

Validating v3 dataset

Schema issues found:
credit_score null rate 15.00% exceeds tolerance
loan_amount values outside expected range
credit_score values outside expected range

Reproducing dataset v1 using pointer

Dataset loaded successfully
Rows: 2000
Columns: 7


##  Task 2 - Experiment Tracking with MLflow

### Scenario
> LoanGuard's data science lead trained three models last sprint - Logistic Regression, Random Forest, and Gradient Boosting. She can't remember which hyperparameters she used for the best one, and the comparison was done in a Slack message that got deleted. You need to run all three as tracked experiments so the decision is transparent and auditable.

### Objective
By the end of this task you will be able to:
- Create an MLflow experiment and log runs with parameters, metrics, and artifacts
- Compare multiple model runs programmatically
- Identify the best model using a business-relevant metric (F1 score for imbalanced default data)
- Log the training dataset version alongside model runs for full traceability

---

###  Background - MLflow Tracking

MLflow Tracking records every experiment run as a **first-class object** with:
- **Parameters** - hyperparameters, config values
- **Metrics** - accuracy, F1, AUC, etc.
- **Artifacts** - model files, plots, data samples
- **Tags** - free-form metadata (dataset version, author, etc.)

Each run gets a unique `run_id` that links it permanently to what produced it.

---

###  Background - MLflow Tracking

MLflow Tracking records every experiment run as a **first-class object** with:
- **Parameters** - hyperparameters, config values
- **Metrics** - accuracy, F1, AUC, etc.
- **Artifacts** - model files, plots, data samples
- **Tags** - free-form metadata (dataset version, author, etc.)

Each run gets a unique `run_id` that links it permanently to what produced it.

---

###  2A - Set up MLflow and prepare data

Configure MLflow tracking and split the versioned dataset into train/test sets.

In [18]:
# ----------------------------------------------------
# BEGINNER 2A - Configure MLflow and prepare data
# ----------------------------------------------------

# Step 1: Configure MLflow tracking to a local SQLite database
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # creates mlflow.db in project folder


# Step 2: Create or select an experiment
experiment = mlflow.set_experiment("loandefault_prediction")
experiment_id = experiment.experiment_id


# Step 3: Load dataset and prepare train/test split
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "employment_years",
    "debt_to_income"
]

TARGET = "default"


# Separate features and target
X = df[FEATURES]
y = df[TARGET]


# Split dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Print dataset statistics
print("Experiment ready")
print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"Default rate (train): {y_train.mean():.1%}")

Experiment ready
Train size : 1,600
Test size  : 400
Default rate (train): 57.6%


###  2B - Log a single MLflow run

Write a function that trains one model and logs everything to MLflow: params, metrics, the model artifact, and a tag for the dataset version.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `with mlflow.start_run(run_name=...) as run:` to start a run. Inside: `mlflow.log_param(key, val)` and `mlflow.log_metric(key, val)`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Log the model with `mlflow.sklearn.log_model(model, artifact_path='model')`. Add tags with `mlflow.set_tag(key, val)`.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy':  accuracy_score(y_te, preds),
            'precision': precision_score(y_te, preds),
            'recall':    recall_score(y_te, preds),
            'f1':        f1_score(y_te, preds),
            'auc':       roc_auc_score(y_te, proba)
        })
        mlflow.set_tag('dataset_version', dataset_version)
        mlflow.set_tag('model_type', model_name)
        mlflow.sklearn.log_model(model, 'model')
        return run.info.run_id, f1_score(y_te, preds)
```

</details>

In [62]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    """
    Train a model and log a complete MLflow run without warnings.

    Logs:
      - params
      - metrics: accuracy, precision, recall, f1, auc
      - model artifact (skops format)
      - tags: dataset_version, model_type

    Returns
    -------
    (run_id : str, f1 : float)
    """
    with mlflow.start_run(run_name=model_name) as run:
        # Train the model
        model.fit(X_tr, y_tr)

        # Predictions & probabilities
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Log parameters
        for key, val in params.items():
            mlflow.log_param(key, val)

        # Compute & log metrics
        metrics = {
            "accuracy": accuracy_score(y_te, preds),
            "precision": precision_score(y_te, preds),
            "recall": recall_score(y_te, preds),
            "f1": f1_score(y_te, preds),
            "auc": roc_auc_score(y_te, proba)
        }
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # Add tags
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)

        #  Log model artifact with skops and explicit pip requirements
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",  # use `name` instead of deprecated artifact_path
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_SKOPS,
            pip_requirements=["scikit-learn>=1.2.2", "skops>=0.13.0"]
        )

        return run.info.run_id, metrics["f1"]


# ---------------------------
# Test the function
# ---------------------------
test_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
test_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}

run_id, f1 = run_experiment(
    test_model,
    model_name="LogisticRegression_test",
    params=test_params,
    X_tr=X_train,
    X_te=X_test,
    y_tr=y_train,
    y_te=y_test,
    dataset_version="v1"
)

print(f"Run ID : {run_id}")
print(f"F1     : {f1:.4f}")

Run ID : f894d8892f03483ab91804903c4cdf18
F1     : 0.8706


###  2C - Run all three candidate models

Run all three model types with the parameters below and collect their run IDs and F1 scores.

In [73]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ---------------------------
# Function to run a single model experiment
# ---------------------------
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Log params
        for k, v in params.items():
            mlflow.log_param(k, v)

        # Log metrics
        metrics = {
            "accuracy": accuracy_score(y_te, preds),
            "precision": precision_score(y_te, preds),
            "recall": recall_score(y_te, preds),
            "f1": f1_score(y_te, preds),
            "auc": roc_auc_score(y_te, proba)
        }
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # Log tags
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)

        # Log model with skops serialization and trusted types
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_SKOPS,
            skops_trusted_types=[
                "sklearn._loss.link.Interval",
                "sklearn._loss.link.LogitLink",
                "sklearn._loss.loss.HalfBinomialLoss"
            ]
        )
        return run.info.run_id, metrics["f1"]

# ---------------------------
# Candidate models
# ---------------------------
candidates = [
    (LogisticRegression(C=0.5, max_iter=1000, random_state=42), "LogisticRegression", {"C":0.5,"max_iter":1000}),
    (RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42), "RandomForest", {"n_estimators":100,"max_depth":6}),
    (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42), "GradientBoosting", {"n_estimators":100,"learning_rate":0.1,"max_depth":4})
]

# ---------------------------
# Run experiments
# ---------------------------
results = []
for model, name, params in candidates:
    run_id, f1 = run_experiment(model, name, params, X_train, X_test, y_train, y_test, "v1")
    results.append({"name": name, "run_id": run_id, "f1": f1})
    print(f"Completed run for {name} | F1 = {f1:.4f}")  # <--- see output immediately

# ---------------------------
# Print summary
# ---------------------------
print("\n" + "═"*55)
print(f"{'Model':<25} {'F1 Score':>10} {'Run ID':>15}")
print("═"*55)
for r in results:
    print(f"{r['name']:<25} {r['f1']:>10.4f}  {r['run_id'][:12]}...")
print("═"*55)

# Best model
best = max(results, key=lambda x: x["f1"])
print(f"\nBest model: {best['name']}  (F1 = {best['f1']:.4f})")
BEST_RUN_ID = best["run_id"]
BEST_MODEL_NAME = best["name"]

2026/03/13 12:51:05 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpowsn65h3/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Completed run for LogisticRegression | F1 = 0.8557


2026/03/13 12:53:07 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpkdpem7kv/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Completed run for RandomForest | F1 = 0.8867


2026/03/13 12:55:10 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp9vx679ti/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Completed run for GradientBoosting | F1 = 0.8976

═══════════════════════════════════════════════════════
Model                       F1 Score          Run ID
═══════════════════════════════════════════════════════
LogisticRegression            0.8557  98b2037d3b05...
RandomForest                  0.8867  c71ab9666937...
GradientBoosting              0.8976  b9b357268434...
═══════════════════════════════════════════════════════

Best model: GradientBoosting  (F1 = 0.8976)


###  2D - Retrieve the best model from MLflow

Use the MLflow client to load the best model back from the tracking server - as if you're another engineer picking up the work.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient()` to interact with the tracking server. `client.get_run(run_id)` retrieves run metadata.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

`mlflow.sklearn.load_model(f'runs:/{run_id}/model')` loads the model artifact from a run.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
run    = client.get_run(BEST_RUN_ID)
print(run.data.metrics)
model  = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
preds  = model.predict(X_test)
```

</details>

In [75]:
#  BEGINNER 2D - Retrieve and verify the best model
# ------------------------------------------------------

from mlflow.tracking import MlflowClient
import mlflow.sklearn
from sklearn.metrics import f1_score

client = MlflowClient()

# Step 1: Retrieve the run metadata for BEST_RUN_ID
best_run = client.get_run(BEST_RUN_ID)


# Step 2: Print what was logged
if best_run:
    print("Run metadata:")
    print(f"  Model type      : {best_run.data.tags.get('model_type')}")
    print(f"  Dataset version : {best_run.data.tags.get('dataset_version')}")
    print(f"  F1 score        : {best_run.data.metrics.get('f1', 0):.4f}")
    print(f"  Params          : {best_run.data.params}")


# Step 3: Load the model artifact
loaded_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")


# Step 4: Verify it produces the same predictions
if loaded_model is not None:
    reloaded_preds = loaded_model.predict(X_test)

    print(f"\nVerification F1: {f1_score(y_test, reloaded_preds):.4f}")
    print("Model successfully retrieved from MLflow")

Run metadata:
  Model type      : GradientBoosting
  Dataset version : v1
  F1 score        : 0.8976
  Params          : {'max_depth': '4', 'learning_rate': '0.1', 'n_estimators': '100'}



Verification F1: 0.8976
Model successfully retrieved from MLflow


In [32]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert mlflow.get_experiment_by_name("loandefault_prediction") is not None, "Create experiment 'loandefault_prediction'"
    passed.append('MLflow experiment exists')
except Exception as e:
    failed.append(('MLflow experiment exists', str(e)))

try:
    assert len(results) == 3, 'results list should have 3 entries'
    passed.append('3 candidates ran')
except Exception as e:
    failed.append(('3 candidates ran', str(e)))

try:
    assert 'BEST_RUN_ID' in dir() or 'BEST_RUN_ID' in globals(), 'BEST_RUN_ID must be defined'
    passed.append('Best run ID set')
except Exception as e:
    failed.append(('Best run ID set', str(e)))

try:
    assert loaded_model is not None, 'loaded_model should not be None'
    passed.append('Model reloads OK')
except Exception as e:
    failed.append(('Model reloads OK', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed:
    print(f'  {p}')

if failed:
    print('\nFailed:')
    for f,e in failed:
        print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 4/4
  MLflow experiment exists
  3 candidates ran
  Best run ID set
  Model reloads OK

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 2

Implement a full experiment comparison pipeline:

**Requirements:**
1. Configure MLflow with the local SQLite URI
2. Write `run_experiment()` that logs: params, 5 metrics (accuracy/precision/recall/f1/auc), model artifact, dataset version tag, and training timestamp tag
3. Run all three candidate models (configurations given in Beginner 2C)
4. Write `find_best_run(experiment_name, metric='f1')` that queries MLflow programmatically (using `MlflowClient.search_runs()`) and returns the run with the highest value of `metric`
5. Load the best model, re-evaluate on test set, and confirm metrics match what was logged
6. **Bonus:** Add a second experiment round where you retrain the best model type on `loans_v2.csv` and log the dataset version change

In [33]:
# ----------------------------------------------------------
# ADVANCED - Task 2 Full Implementation
# ----------------------------------------------------------

import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# ----------------------------------------------------------
#  Configure MLflow
# ----------------------------------------------------------

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("loandefault_prediction")
# ----------------------------------------------------------
#  Load Dataset
# ----------------------------------------------------------

df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "employment_years",
    "debt_to_income"
]

TARGET = "default"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
# ----------------------------------------------------------
#  Experiment Runner
# ----------------------------------------------------------

def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):

    with mlflow.start_run(run_name=model_name) as run:

        # Train model
        model.fit(X_tr, y_tr)

        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:,1]

        # Metrics
        accuracy  = accuracy_score(y_te, preds)
        precision = precision_score(y_te, preds)
        recall    = recall_score(y_te, preds)
        f1        = f1_score(y_te, preds)
        auc       = roc_auc_score(y_te, proba)

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metrics({
            "accuracy":accuracy,
            "precision":precision,
            "recall":recall,
            "f1":f1,
            "auc":auc
        })

        # Tags
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("training_timestamp", str(datetime.now()))

        # Save model
        mlflow.sklearn.log_model(model,"model")

        return run.info.run_id, f1
# ----------------------------------------------------------
#  Run 3 Candidate Models
# ----------------------------------------------------------

candidates = [

    (
        LogisticRegression(C=0.5,max_iter=1000,random_state=42),
        "LogisticRegression",
        {"model":"LogisticRegression","C":0.5,"max_iter":1000}
    ),

    (
        RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42),
        "RandomForest",
        {"model":"RandomForest","n_estimators":100,"max_depth":6}
    ),

    (
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=42
        ),
        "GradientBoosting",
        {
            "model":"GradientBoosting",
            "n_estimators":100,
            "learning_rate":0.1,
            "max_depth":4
        }
    )
]
results = []

for model, name, params in candidates:

    run_id, f1 = run_experiment(
        model,
        name,
        params,
        X_train,
        X_test,
        y_train,
        y_test,
        dataset_version="v1"
    )

    results.append({
        "name":name,
        "run_id":run_id,
        "f1":f1
    })
print("\nModel comparison")

for r in results:
    print(r)
# ----------------------------------------------------------
#  Find Best Run Programmatically
# ----------------------------------------------------------

def find_best_run(experiment_name, metric="f1"):

    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    best_run = None
    best_score = -1

    for r in runs:

        score = r.data.metrics.get(metric,0)

        if score > best_score:
            best_score = score
            best_run = r

    return best_run


best_run = find_best_run("loandefault_prediction")

print("\nBest Run ID:", best_run.info.run_id)
print("Best F1:", best_run.data.metrics["f1"])
# ----------------------------------------------------------
#  Load Best Model
# ----------------------------------------------------------

best_model = mlflow.sklearn.load_model(
    f"runs:/{best_run.info.run_id}/model"
)

preds = best_model.predict(X_test)

print("\nVerification F1:", f1_score(y_test,preds))
# ----------------------------------------------------------
#  BONUS – Retrain Best Model on loans_v2
# ----------------------------------------------------------

df2 = pd.read_csv(LAB_DIR / "data" / "loans_v2.csv").dropna()

X2 = df2[FEATURES]
y2 = df2[TARGET]

X_train2,X_test2,y_train2,y_test2 = train_test_split(
    X2,y2,
    test_size=0.2,
    random_state=42
)


best_model_type = best_run.data.tags["model_type"]

if best_model_type == "RandomForest":
    model = RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42)

elif best_model_type == "GradientBoosting":
    model = GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42)

else:
    model = LogisticRegression(C=0.5,max_iter=1000,random_state=42)


run_experiment(
    model,
    best_model_type+"_v2",
    {"retrain_dataset":"v2"},
    X_train2,
    X_test2,
    y_train2,
    y_test2,
    dataset_version="v2"
)

print("\nSecond experiment round completed on dataset v2")

2026/03/12 15:09:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 15:09:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/12 15:09:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 15:09:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p


Model comparison
{'name': 'LogisticRegression', 'run_id': '57c85ca8b1854b019ec2785a134d127e', 'f1': 0.8706365503080082}
{'name': 'RandomForest', 'run_id': '03168eb2050742b49cf63c875d3ed4a9', 'f1': 0.8747433264887063}
{'name': 'GradientBoosting', 'run_id': 'ab4e56d35ce24977a715571c03dc9f8b', 'f1': 0.8722109533468559}

Best Run ID: 03168eb2050742b49cf63c875d3ed4a9
Best F1: 0.8747433264887063



Verification F1: 0.8747433264887063


2026/03/12 15:09:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 15:09:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Second experiment round completed on dataset v2


---
# TASK 3 - Model Versioning
> Give your model a name that tells the whole story
---

##  Task 3 - Model Versioning - Naming, Metadata & Semantic Versions

### Scenario
> LoanGuard's best model from Task 2 needs to be packaged for the team. The previous engineer just saved files as `model.pkl` with no metadata. When it broke, nobody knew what version was running or how to go back. You need to create a versioned model artifact with full metadata and implement semantic versioning logic.

### Objective
By the end of this task you will be able to:
- Save a model artifact with complete metadata (dataset version, metrics, git hash, author)
- Implement semantic versioning logic (MAJOR.MINOR.PATCH) for model releases
- Register a named, versioned model in MLflow Model Registry
- Retrieve a specific model version by name

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  3A - Save a model with complete metadata

Save the best model from Task 2 alongside a metadata sidecar file that fully describes it.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Save the model with `pickle.dump()`. Create a separate `_metadata.json` file in the same folder with all relevant context.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Metadata should include: model_name, version, dataset_version, metrics, params, trained_at, and a simulated git_hash (`hashlib.md5(str(time.time()).encode()).hexdigest()[:8]`).

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def save_versioned_model(model, name, version, metrics, params, dataset_version):
    path = LAB_DIR / 'models' / f'{name}-{version}'
    path.mkdir(exist_ok=True)
    with open(path / 'model.pkl', 'wb') as f:
        pickle.dump(model, f)
    meta = {
        'name': name, 'version': version,
        'dataset_version': dataset_version,
        'metrics': metrics, 'params': params,
        'git_hash': hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        'trained_at': datetime.now().isoformat(),
        'mlflow_run_id': BEST_RUN_ID
    }
    with open(path / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    return path, meta
```

</details>

In [34]:
# ------------------------------------------------------------
# BEGINNER 3A - Save versioned model with metadata
# ------------------------------------------------------------

import pickle
import json
import hashlib
import time
from datetime import datetime
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient


def save_versioned_model(model, name, version, metrics, params,
                          dataset_version, run_id):

    """
    Save a model and its complete metadata sidecar.
    """

    # Step 1: Create output directory
    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(parents=True, exist_ok=True)


    # Step 2: Save model with pickle
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)


    # Step 3: Build metadata dict
    metadata = {
        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": "loanGuard-ml-team"
    }


    # Step 4: Save metadata as JSON
    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)


    return path, metadata


# ------------------------------------------------------------
# Save the best model from Task 2
# ------------------------------------------------------------

best_run_data = MlflowClient().get_run(BEST_RUN_ID)

best_metrics = best_run_data.data.metrics
best_params  = best_run_data.data.params

best_fitted = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")


model_path, model_meta = save_versioned_model(
    model           = best_fitted,
    name            = "loan-default-detector",
    version         = "v1.0.0",
    metrics         = best_metrics,
    params          = best_params,
    dataset_version = "v1",
    run_id          = BEST_RUN_ID
)


print(f"Saved to: {model_path}")

print("\nMetadata:")
print(json.dumps(model_meta, indent=2))

Saved to: loanGuard_lab/models/loan-default-detector-v1.0.0

Metadata:
{
  "name": "loan-default-detector",
  "version": "v1.0.0",
  "dataset_version": "v1",
  "metrics": {
    "accuracy": 0.8475,
    "precision": 0.8623481781376519,
    "recall": 0.8875,
    "f1": 0.8747433264887063,
    "auc": 0.9296354166666667
  },
  "params": {
    "model": "RandomForest",
    "n_estimators": "100",
    "max_depth": "6"
  },
  "git_hash": "8736ea99",
  "trained_at": "2026-03-12T15:26:49.132206",
  "mlflow_run_id": "b7cd856c2e934e928cf7edf6adbc1098",
  "author": "loanGuard-ml-team"
}


###  3B - Semantic versioning logic

Implement a function that determines the correct next version number given the type of change.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Parse the version string by splitting on '.': `major, minor, patch = version.lstrip('v').split('.')`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For 'major' bump: increment major, reset minor and patch to 0. For 'minor': increment minor, reset patch. For 'patch': just increment patch.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def bump_version(current, bump_type):
    major, minor, patch = [int(x) for x in current.lstrip('v').split('.')]
    if bump_type == 'major':   return f'v{major+1}.0.0'
    elif bump_type == 'minor': return f'v{major}.{minor+1}.0'
    elif bump_type == 'patch': return f'v{major}.{minor}.{patch+1}'
```

</details>

In [35]:
# ------------------------------------------------------------
# BEGINNER 3B - Semantic versioning
# ------------------------------------------------------------

def bump_version(current_version, bump_type):

    """
    Increment a semantic version string.
    """

    # Step 1: Parse version string
    major, minor, patch = [int(x) for x in current_version.lstrip("v").split(".")]

    # Step 2: Apply bump logic
    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----- Test all bump types -----
start = "v1.0.0"

print(f"Start version      : {start}")
print(f"After patch bump   : {bump_version(start, 'patch')}   ← threshold tweak")
print(f"After minor bump   : {bump_version(start, 'minor')}   ← new training data")
print(f"After major bump   : {bump_version(start, 'major')}   ← new architecture")
print()


# Simulate LoanGuard version history
history = [
    ("v1.0.0", "Initial release - Random Forest on loans_v1"),
    (bump_version("v1.0.0", "patch"), "Fixed decision threshold from 0.5 → 0.46"),
    (bump_version("v1.0.1", "minor"), "Retrained on loans_v2 with new num_accounts feature"),
    (bump_version("v1.1.0", "major"), "Switched to GradientBoosting - breaking output format change"),
]

print("LoanGuard model version history:")

for ver, desc in history:
    print(f"  {ver:<12} {desc}")

Start version      : v1.0.0
After patch bump   : v1.0.1   ← threshold tweak
After minor bump   : v1.1.0   ← new training data
After major bump   : v2.0.0   ← new architecture

LoanGuard model version history:
  v1.0.0       Initial release - Random Forest on loans_v1
  v1.0.1       Fixed decision threshold from 0.5 → 0.46
  v1.1.0       Retrained on loans_v2 with new num_accounts feature
  v2.0.0       Switched to GradientBoosting - breaking output format change


In [36]:
# ----------------------------------------------------------
# BEGINNER 3C - Register model in MLflow Registry
# ----------------------------------------------------------

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import f1_score

REGISTRY_NAME = "LoanDefaultDetector"

# Step 1: Register the best model run in the MLflow registry
model_uri = f"runs:/{BEST_RUN_ID}/model"

registered = mlflow.register_model(model_uri, REGISTRY_NAME)

if registered:
    print(f"Registered model : {registered.name}")
    print(f"Version          : {registered.version}")
    print(f"Source run       : {registered.source}")


# Step 2: Retrieve and inspect all registered versions
client = MlflowClient()

versions = client.get_latest_versions(REGISTRY_NAME)

print("\nRegistered versions:")
for v in versions:
    print(f"Version {v.version} | Stage: {v.current_stage} | Run ID: {v.run_id}")


# Step 3: Load the registered model by name and verify
latest_version = versions[0].version

registry_model = mlflow.sklearn.load_model(
    f"models:/{REGISTRY_NAME}/{latest_version}"
)

if registry_model is not None:
    verify_preds = registry_model.predict(X_test)

    print(f"\nVerification F1 from registry: {f1_score(y_test, verify_preds):.4f}")
    print("Model successfully retrieved from registry by name")

Successfully registered model 'LoanDefaultDetector'.
2026/03/12 15:32:48 WARNING mlflow.tracking._model_registry.fluent: Run with id b7cd856c2e934e928cf7edf6adbc1098 has no artifacts at artifact path 'model', registering model based on models:/m-b34868f5bfea4421939fbbb6f1e5be22 instead


Registered model : LoanDefaultDetector
Version          : 1
Source run       : models:/m-b34868f5bfea4421939fbbb6f1e5be22

Registered versions:
Version 1 | Stage: None | Run ID: b7cd856c2e934e928cf7edf6adbc1098

Verification F1 from registry: 0.8747
Model successfully retrieved from registry by name


Created version '1' of model 'LoanDefaultDetector'.


In [37]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'model.pkl').exists(), 'model.pkl not found'
    passed.append('Versioned model saved')
except Exception as e:
    failed.append(('Versioned model saved', str(e)))

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'metadata.json').exists(), 'metadata.json not found'
    passed.append('Metadata saved')
except Exception as e:
    failed.append(('Metadata saved', str(e)))

try:
    assert bump_version('v1.0.0', 'patch') == 'v1.0.1', 'patch bump failed'
    passed.append('Patch bump correct')
except Exception as e:
    failed.append(('Patch bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'minor') == 'v1.1.0', 'minor bump failed'
    passed.append('Minor bump correct')
except Exception as e:
    failed.append(('Minor bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'major') == 'v2.0.0', 'major bump failed'
    passed.append('Major bump correct')
except Exception as e:
    failed.append(('Major bump correct', str(e)))

try:
    assert registered is not None, 'registered model should not be None'
    passed.append('Model registered')
except Exception as e:
    failed.append(('Model registered', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 6/6
  Versioned model saved
  Metadata saved
  Patch bump correct
  Minor bump correct
  Major bump correct
  Model registered

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 3

Build a complete model versioning and registry system:

**Requirements:**
1. `save_versioned_model()` - saves model + full metadata sidecar (all fields from Beginner 3A)
2. `bump_version()` - semantic version logic with validation (reject unknown bump types)
3. `ModelVersionHistory` class with methods:
   - `record(version, change_type, description, metrics)` - adds an entry
   - `get_changelog()` - returns formatted changelog string
   - `get_version_for_rollback(n_versions_back)` - returns the version string from n releases ago
4. Register model in MLflow Registry, add a description, and add the semantic version as a tag
5. **Demonstrate rollback:** Simulate a production failure, use `get_version_for_rollback(1)` to identify the previous version, load it from the local store, and confirm it produces different (better) metrics than the failed version

In [38]:
# ----------------------------------------------------------
# ADVANCED TASK 3 - Complete Model Versioning System
# ---------------------------------------------------------
import pickle
import json
import hashlib
import time
from datetime import datetime
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.metrics import f1_score
# ----------------------------------------------------------
#  Save Versioned Model
# ----------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params,
                         dataset_version, run_id, author="ml-team"):

    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(parents=True, exist_ok=True)

    # Save model
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    metadata = {

        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": author
    }

    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata
# ----------------------------------------------------------
#  Semantic Versioning Logic
# ----------------------------------------------------------

def bump_version(current, bump_type):

    major, minor, patch = [int(x) for x in current.lstrip("v").split(".")]

    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    else:
        raise ValueError("Invalid bump type. Use major/minor/patch")

    return f"v{major}.{minor}.{patch}"
# ----------------------------------------------------------
# Version History Manager
# ----------------------------------------------------------

class ModelVersionHistory:

    def __init__(self):

        self.history = []

    def record(self, version, change_type, description, metrics):

        entry = {
            "version": version,
            "change_type": change_type,
            "description": description,
            "metrics": metrics,
            "timestamp": datetime.now().isoformat()
        }

        self.history.append(entry)


    def get_changelog(self):

        lines = []

        for h in self.history:

            line = (
                f"{h['version']} | {h['change_type']} | "
                f"{h['description']} | F1={h['metrics'].get('f1',0):.4f}"
            )

            lines.append(line)

        return "\n".join(lines)


    def get_version_for_rollback(self, n_versions_back):

        if n_versions_back >= len(self.history):
            raise ValueError("Rollback exceeds version history")

        return self.history[-(n_versions_back+1)]["version"]
# ----------------------------------------------------------
#  Initialize Version History
# ----------------------------------------------------------
history = ModelVersionHistory()
# ----------------------------------------------------------
# Save Current Best Model
# ----------------------------------------------------------
client = MlflowClient()
run = client.get_run(BEST_RUN_ID)
metrics = run.data.metrics
params = run.data.params
best_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
version = "v1.0.0"
path, metadata = save_versioned_model(

    best_model,
    "loan-default-detector",
    version,
    metrics,
    params,
    "v1",
    BEST_RUN_ID
)
history.record(

    version,
    "major",
    "Initial production release",
    metrics
)

print("Model saved:", path)
# ----------------------------------------------------------
#  Register in MLflow Model Registry
# ----------------------------------------------------------

MODEL_NAME = "LoanDefaultDetector"

model_uri = f"runs:/{BEST_RUN_ID}/model"

registered = mlflow.register_model(model_uri, MODEL_NAME)

client.update_registered_model(

    MODEL_NAME,
    description="Loan default prediction model used in LoanGuard system"
)

client.set_model_version_tag(

    MODEL_NAME,
    registered.version,
    "semantic_version",
    version
)

print("Registered version:", registered.version)
# ---------------------------------------------------------
# Simulate New Version with Worse Performance
# ---------------------------------------------------------
new_version = bump_version(version, "patch")

failed_metrics = {"f1": 0.61}

history.record(

    new_version,
    "patch",
    "Threshold tuning caused performance drop",
    failed_metrics
)
print("\nNew version deployed:", new_version)
print("Production failure detected")
# ----------------------------------------------------------
# Rollback Logic
# ----------------------------------------------------------
rollback_version = history.get_version_for_rollback(1)

print("Rollback target:", rollback_version)
# ----------------------------------------------------------
# Load Rolled Back Model
# ----------------------------------------------------------

rollback_path = LAB_DIR / "models" / f"loan-default-detector-{rollback_version}"

with open(rollback_path / "model.pkl", "rb") as f:
    rollback_model = pickle.load(f)


preds = rollback_model.predict(X_test)

rollback_f1 = f1_score(y_test, preds)

print("\nRollback model F1:", rollback_f1)
print("Rollback successful")
# ----------------------------------------------------------
#  Print Changelog
# ----------------------------------------------------------

print("\nMODEL CHANGELOG")

print(history.get_changelog())

Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/12 15:42:02 WARNING mlflow.tracking._model_registry.fluent: Run with id b7cd856c2e934e928cf7edf6adbc1098 has no artifacts at artifact path 'model', registering model based on models:/m-b34868f5bfea4421939fbbb6f1e5be22 instead


Model saved: loanGuard_lab/models/loan-default-detector-v1.0.0
Registered version: 2

New version deployed: v1.0.1
Production failure detected
Rollback target: v1.0.0

Rollback model F1: 0.8747433264887063
Rollback successful

MODEL CHANGELOG
v1.0.0 | major | Initial production release | F1=0.8747
v1.0.1 | patch | Threshold tuning caused performance drop | F1=0.6100


Created version '2' of model 'LoanDefaultDetector'.


---
# TASK 4 - Registry Lifecycle Management
> Staging → Production → Archived: govern your models
---

##  Task 4 - Registry Lifecycle - Staging, Production, Archived

### Scenario
> LoanGuard just hired a new senior ML engineer. She asks: 'What model is in production right now? When was it approved? Who approved it? What does rollback look like?' Nobody can answer. In this task you'll implement the full governance lifecycle - every model moves through defined stages with approvals and audit logs.

### Objective
By the end of this task you will be able to:
- Transition a model through Staging → Production → Archived stages in MLflow
- Write an audit log that records every lifecycle event
- Implement a rollback function that demotes the current production model
- Query the registry to answer governance questions programmatically

---

###  4A - Transition model through lifecycle stages

Move the registered model through the stages: None → Staging → Production

In [47]:
# ---------------------------------------------------------
# BEGINNER 4A - Lifecycle stage transitions
# ---------------------------------------------------------

from mlflow.tracking import MlflowClient

client = MlflowClient()

def transition_stage(name, version, stage, reason=""):

    """
    Transition a registered model version to a new stage
    and record the reason.
    """

    # Step 1: Transition the model stage
    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=stage,
        archive_existing_versions=False
    )

    # Step 2: Update the model version description
    if reason:
        client.update_model_version(
            name=name,
            version=version,
            description=reason
        )

    print(f"{name} v{version}  →  {stage}")

    if reason:
        print(f"Reason: {reason}")


# ----- Walk the model through its lifecycle -----

latest = client.get_latest_versions(REGISTRY_NAME)

if latest:

    ver = latest[0].version

    print("Stage transitions for LoanDefaultDetector:")
    print("─" * 50)

    # Move to Staging
    transition_stage(
        REGISTRY_NAME,
        ver,
        "Staging",
        reason="Passed unit tests and offline evaluation - F1 > 0.72 threshold met"
    )

    # Move to Production
    transition_stage(
        REGISTRY_NAME,
        ver,
        "Production",
        reason="Approved by ML review board after A/B testing and validation on latest loan dataset"
    )

    # Verify stage
    current = client.get_model_version(REGISTRY_NAME, ver)

    print(f"\nCurrent stage: {current.current_stage}")
    print(f"Description  : {current.description}")

Stage transitions for LoanDefaultDetector:
──────────────────────────────────────────────────
LoanDefaultDetector v2  →  Staging
Reason: Passed unit tests and offline evaluation - F1 > 0.72 threshold met
LoanDefaultDetector v2  →  Production
Reason: Approved by ML review board after A/B testing and validation on latest loan dataset

Current stage: Production
Description  : Approved by ML review board after A/B testing and validation on latest loan dataset


###  4B - Build an audit log

Every lifecycle event should be recorded. Implement an audit log that tracks: timestamp, model name, version, old stage, new stage, and the actor/reason.

In [48]:
# ---------------------------------------------------------
# BEGINNER 4B - Audit log
# ---------------------------------------------------------
import json
from datetime import datetime

AUDIT_LOG = []   # list of event dicts


def log_audit_event(model_name, version, from_stage, to_stage,
                     actor, reason):

    """
    Record a lifecycle event in the audit log and persist to disk.
    """

    event = {
        "timestamp":    datetime.now().isoformat(),
        "model_name":   model_name,
        "version":      version,
        "from_stage":   from_stage,
        "to_stage":     to_stage,
        "actor":        actor,
        "reason":       reason
    }

    # Step 1: Append to in-memory log
    AUDIT_LOG.append(event)

    # Step 2: Save to JSON file (append-style)
    log_path = LAB_DIR / "audit_log.json"

    if log_path.exists():
        with open(log_path, "r") as f:
            data = json.load(f)
    else:
        data = []

    data.append(event)

    with open(log_path, "w") as f:
        json.dump(data, f, indent=2)

    return event

def print_audit_log():

    """Print a formatted audit log table."""

    if not AUDIT_LOG:
        print("Audit log is empty")
        return

    print(f"{'Timestamp':<26} {'Model':<22} {'v':<4} {'From':<12} {'To':<12} {'Actor':<18} Reason")
    print("─" * 110)

    for e in AUDIT_LOG:

        ts = e['timestamp'][:19]

        print(
            f"{ts:<26} {e['model_name']:<22} {e['version']:<4} "
            f"{e['from_stage']:<12} {e['to_stage']:<12} "
            f"{e['actor']:<18} {e['reason']}"
        )

# ----- Replay lifecycle events through audit log -----

ver = client.get_latest_versions(REGISTRY_NAME)[0].version

log_audit_event(
    REGISTRY_NAME,
    ver,
    "None",
    "Staging",
    "ci-system",
    "Automated: offline eval passed F1 threshold"
)

log_audit_event(
    REGISTRY_NAME,
    ver,
    "Staging",
    "Production",
    "ml-lead@loanGuard.ai",
    "Manual approval: A/B test shows +3% approval lift"
)

print("LoanGuard Model Audit Log:")
print("=" * 110)

print_audit_log()

LoanGuard Model Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-12T16:27:43        LoanDefaultDetector    1    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-12T16:27:43        LoanDefaultDetector    1    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift


###  4C - Rollback

Simulate a production failure: a new model version is promoted, starts failing, and you need to roll back to the previous production model.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

To rollback: archive the current production version, then transition the previous version back to Production.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Use `client.search_model_versions(f"name='{name}'")` to find all versions. Filter by `current_stage == 'Archived'` to find previous production candidates.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def rollback_production(name, rollback_to_version, reason):
    # 1. Find and archive current production version
    current_prod = [v for v in client.search_model_versions(f"name='{name}'")
                    if v.current_stage == 'Production']
    if current_prod:
        client.transition_model_version_stage(name, current_prod[0].version, 'Archived')
    # 2. Promote rollback target to Production
    client.transition_model_version_stage(name, rollback_to_version, 'Production')
```

</details>

In [51]:
# ---------------------------------------------------------
# BEGINNER 4C - Simulate promotion of v2, then rollback
# ---------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from datetime import datetime
import json

client = MlflowClient()

# ---------------------------
# Simulating a bad deployment
# ---------------------------

print("Simulating a bad deployment...")

weak_model = LogisticRegression(C=0.001, max_iter=100, random_state=42)

with mlflow.start_run(run_name="WeakModel_bad_deploy") as bad_run:

    weak_model.fit(X_train, y_train)

    weak_preds = weak_model.predict(X_test)

    mlflow.log_metric("f1", f1_score(y_test, weak_preds))
    mlflow.set_tag("dataset_version", "v1")

    mlflow.sklearn.log_model(weak_model, "model")

    bad_run_id = bad_run.info.run_id


# ---------------------------
# Register bad model (v2)
# ---------------------------

bad_registered = mlflow.register_model(
    f"runs:/{bad_run_id}/model",
    REGISTRY_NAME
)

bad_version = bad_registered.version


# Promote weak model to production
client.transition_model_version_stage(
    REGISTRY_NAME,
    bad_version,
    "Production",
    archive_existing_versions=True
)

log_audit_event(
    REGISTRY_NAME,
    bad_version,
    "Staging",
    "Production",
    "auto-deploy-pipeline",
    "Automated promotion - metric check bypassed (BUG)"
)

print(f"Bad model (version {bad_version}) promoted to Production - F1: {f1_score(y_test, weak_preds):.4f}")
print("Production is now degraded!")

print()
print("─" * 50)
print("Initiating rollback...")
print("─" * 50)
# ---------------------------
# Rollback Function
# ---------------------------

def rollback_production(registry_name, rollback_to_version, reason, actor):

    # Step 1: Find current production version
    current_prod_versions = [
        v for v in client.search_model_versions(f"name='{registry_name}'")
        if v.current_stage == "Production"
    ]

    # Step 2: Archive current production
    for v in current_prod_versions:

        client.transition_model_version_stage(
            name=registry_name,
            version=v.version,
            stage="Archived"
        )

        log_audit_event(
            registry_name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: replaced by v{rollback_to_version}. {reason}"
        )

    # Step 3: Promote rollback target
    client.transition_model_version_stage(
        name=registry_name,
        version=rollback_to_version,
        stage="Production"
    )

    log_audit_event(
        registry_name,
        rollback_to_version,
        "Archived",
        "Production",
        actor,
        f"ROLLBACK TARGET: {reason}"
    )

    print(f"Rolled back to version {rollback_to_version}")
# ---------------------------
# Execute rollback
# ---------------------------
v1_version = client.get_latest_versions(REGISTRY_NAME, stages=["Archived"])

if v1_version:

    rollback_production(
        registry_name       = REGISTRY_NAME,
        rollback_to_version = v1_version[0].version,
        reason              = "Production F1 dropped from 0.72 to 0.41 after v2 deployment",
        actor               = "ml-lead@loanGuard.ai"
    )


print()
print("Final Audit Log:")
print("=" * 110)
print_audit_log()

Simulating a bad deployment...


2026/03/12 16:34:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 16:34:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/12 16:34:04 WARNING mlflow.tracking._model_registry.fluent: Run with id bb9e4a19840c49fa8fa57597e32b95ed has no artifacts at artifact path 'model', registering model based on models:/m-8ade130cb00b49ebba1d654e6526d54c instead


Bad model (version 3) promoted to Production - F1: 0.6973
Production is now degraded!

──────────────────────────────────────────────────
Initiating rollback...
──────────────────────────────────────────────────
Rolled back to version 2

Final Audit Log:
Timestamp                  Model                  v    From         To           Actor              Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
2026-03-12T16:27:43        LoanDefaultDetector    1    None         Staging      ci-system          Automated: offline eval passed F1 threshold
2026-03-12T16:27:43        LoanDefaultDetector    1    Staging      Production   ml-lead@loanGuard.ai Manual approval: A/B test shows +3% approval lift
2026-03-12T16:34:04        LoanDefaultDetector    3    Staging      Production   auto-deploy-pipeline Automated promotion - metric check bypassed (BUG)
2026-03-12T16:34:04        LoanDefaultDetector    3    Production   Archived   

Created version '3' of model 'LoanDefaultDetector'.


###  4D - Governance Query

Answer the governance questions that LoanGuard's new engineer asked - programmatically.

In [53]:
# ------------------------------------------------------------
# BEGINNER 4D - Answer governance questions programmatically
# ------------------------------------------------------------

print("╔══════════════════════════════════════════════════════╗")
print("║        LoanGuard Model Governance Report             ║")
print("╚══════════════════════════════════════════════════════╝")
print()


# Q1: What model is in production right now?

prod_models = [
    v for v in client.search_model_versions(f"name='{REGISTRY_NAME}'")
    if v.current_stage == "Production"
]

print("Q1 - Current production model:")

for m in prod_models:
    print(f"     {m.name}  version {m.version}  ({m.current_stage})")

print()


# Q2: When was it promoted to production?

prod_event = None

for e in reversed(AUDIT_LOG):
    if e["to_stage"] == "Production":
        prod_event = e
        break

if prod_event:
    print(f"Q2 - Promoted to production at: {prod_event['timestamp']}")

print()


# Q3: Who approved it?

if prod_event:
    print(f"Q3 - Approved by: {prod_event['actor']}")

print()


# Q4: What data trained it?

if prod_models:

    prod_model = prod_models[0]

    run_id = prod_model.run_id

    run_info = client.get_run(run_id)

    dataset_version = run_info.data.tags.get("dataset_version", "unknown")

    print(f"Q4 - Training dataset: {dataset_version}")

print()


# Q5: What were the production model's metrics?

if prod_models:

    run_id = prod_models[0].run_id

    run_info = client.get_run(run_id)

    metrics = run_info.data.metrics

    print("Q5 - Metrics at registration:")

    for k, v in metrics.items():
        print(f"     {k} : {v:.4f}")

╔══════════════════════════════════════════════════════╗
║        LoanGuard Model Governance Report             ║
╚══════════════════════════════════════════════════════╝

Q1 - Current production model:
     LoanDefaultDetector  version 1  (Production)

Q2 - Promoted to production at: 2026-03-12T16:34:04.257041

Q3 - Approved by: ml-lead@loanGuard.ai

Q4 - Training dataset: v1

Q5 - Metrics at registration:
     accuracy : 0.8475
     precision : 0.8623
     recall : 0.8875
     f1 : 0.8747
     auc : 0.9296


In [54]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert len(AUDIT_LOG) >= 2, 'Audit log must have at least 2 events'
    passed.append('Audit log has entries')
except Exception as e:
    failed.append(('Audit log has entries', str(e)))

try:
    assert all('timestamp' in e for e in AUDIT_LOG), 'Each event needs a timestamp'
    passed.append('Audit log has timestamps')
except Exception as e:
    failed.append(('Audit log has timestamps', str(e)))

try:
    assert (LAB_DIR / 'audit_log.json').exists(), 'audit_log.json not written to disk'
    passed.append('Audit log persisted')
except Exception as e:
    failed.append(('Audit log persisted', str(e)))

try:
    assert len(client.get_latest_versions(REGISTRY_NAME, stages=['Production'])) > 0, 'No model in Production stage'
    passed.append('Production model exists')
except Exception as e:
    failed.append(('Production model exists', str(e)))

try:
    assert len([e for e in AUDIT_LOG if 'ROLLBACK' in e.get('reason','')]) > 0, 'No rollback event in audit log'
    passed.append('Rollback executed')
except Exception as e:
    failed.append(('Rollback executed', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  Audit log has entries
  Audit log has timestamps
  Audit log persisted
  Production model exists
  Rollback executed

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 4

Build a complete governance system:

**Requirements:**
1. `transition_stage()` with stage validation (only allow valid stage progressions: None→Staging, Staging→Production, Production→Archived)
2. `log_audit_event()` that writes to both in-memory list and a JSON file on disk
3. `rollback_production()` - archives current prod, promotes target, logs both events
4. `GovernanceReport` class with methods:
   - `current_production()` - returns name, version, stage, promoter, promotion time
   - `full_changelog(model_name)` - returns complete event history from audit log
   - `models_in_stage(stage)` - returns all models currently in a given stage
5. Simulate the full LoanGuard lifecycle: register v1 → staging → production → register bad v2 → production → rollback → archive v2
6. Print the final governance report answering all 5 questions from 4D

**Bonus:** Add a gate that blocks promotion to Production if the model's F1 score (from MLflow metrics) is below 0.65

In [55]:
import json
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

AUDIT_LOG = []
LOG_PATH = LAB_DIR / "audit_log.json"

VALID_TRANSITIONS = {
    "None": ["Staging"],
    "Staging": ["Production"],
    "Production": ["Archived"]
}
# --------------------------------------------------
# Audit Logging
# --------------------------------------------------

def log_audit_event(model_name, version, from_stage, to_stage, actor, reason):

    event = {
        "timestamp": datetime.now().isoformat(),
        "model_name": model_name,
        "version": version,
        "from_stage": from_stage,
        "to_stage": to_stage,
        "actor": actor,
        "reason": reason
    }

    AUDIT_LOG.append(event)

    if LOG_PATH.exists():
        with open(LOG_PATH, "r") as f:
            data = json.load(f)
    else:
        data = []

    data.append(event)

    with open(LOG_PATH, "w") as f:
        json.dump(data, f, indent=2)

    return event
# --------------------------------------------------
# Stage Transition with Validation + Gate
# --------------------------------------------------

def transition_stage(name, version, new_stage, actor, reason):

    mv = client.get_model_version(name, version)
    current_stage = mv.current_stage or "None"

    if new_stage not in VALID_TRANSITIONS.get(current_stage, []):
        raise Exception(f"Invalid transition {current_stage} → {new_stage}")

    # BONUS: F1 Gate
    if new_stage == "Production":

        run = client.get_run(mv.run_id)
        f1 = run.data.metrics.get("f1", 0)

        if f1 < 0.65:
            raise Exception(f"Blocked promotion: F1={f1:.3f} < 0.65")

    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=new_stage
    )

    log_audit_event(name, version, current_stage, new_stage, actor, reason)
# --------------------------------------------------
# Rollback System
# --------------------------------------------------

def rollback_production(name, target_version, actor, reason):

    prod_models = [
        v for v in client.search_model_versions(f"name='{name}'")
        if v.current_stage == "Production"
    ]

    for v in prod_models:

        client.transition_model_version_stage(
            name=name,
            version=v.version,
            stage="Archived"
        )

        log_audit_event(
            name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: {reason}"
        )

    client.transition_model_version_stage(
        name=name,
        version=target_version,
        stage="Production"
    )

    log_audit_event(
        name,
        target_version,
        "Archived",
        "Production",
        actor,
        "Rollback target promoted"
    )
# --------------------------------------------------
# Governance Report Class
# --------------------------------------------------
class GovernanceReport:

    def current_production(self):

        prod = [
            v for v in client.search_model_versions(f"name='{REGISTRY_NAME}'")
            if v.current_stage == "Production"
        ]

        if not prod:
            return None

        version = prod[0].version

        event = next(
            e for e in reversed(AUDIT_LOG)
            if e["version"] == version and e["to_stage"] == "Production"
        )

        return {
            "name": REGISTRY_NAME,
            "version": version,
            "stage": "Production",
            "promoter": event["actor"],
            "time": event["timestamp"]
        }

    def full_changelog(self, model_name):

        return [
            e for e in AUDIT_LOG
            if e["model_name"] == model_name
        ]

    def models_in_stage(self, stage):

        return [
            v for v in client.search_model_versions(f"name='{REGISTRY_NAME}'")
            if v.current_stage == stage
        ]
# -------------------------------------------------
# Simulate LoanGuard Lifecycle
# --------------------------------------------------
print("Simulating lifecycle...")

# --- Train GOOD MODEL (v1)

good_model = LogisticRegression(max_iter=500)

with mlflow.start_run(run_name="LoanGuard_v1") as run:

    good_model.fit(X_train, y_train)

    preds = good_model.predict(X_test)
    f1 = f1_score(y_test, preds)

    mlflow.log_metric("f1", f1)
    mlflow.set_tag("dataset_version", "v1")

    mlflow.sklearn.log_model(good_model, "model")

    run_id = run.info.run_id


v1 = mlflow.register_model(f"runs:/{run_id}/model", REGISTRY_NAME).version

transition_stage(REGISTRY_NAME, v1, "Staging", "ci-system", "Initial validation passed")
transition_stage(REGISTRY_NAME, v1, "Production", "ml-lead", "Approved for deployment")
# --- Train BAD MODEL (v2)

bad_model = LogisticRegression(C=0.001)

with mlflow.start_run(run_name="LoanGuard_v2_bad") as run:

    bad_model.fit(X_train, y_train)

    preds = bad_model.predict(X_test)
    f1_bad = f1_score(y_test, preds)

    mlflow.log_metric("f1", f1_bad)
    mlflow.set_tag("dataset_version", "v2")

    mlflow.sklearn.log_model(bad_model, "model")

    bad_run_id = run.info.run_id


v2 = mlflow.register_model(f"runs:/{bad_run_id}/model", REGISTRY_NAME).version
# Attempt promotion (may fail due to gate)

try:
    transition_stage(REGISTRY_NAME, v2, "Staging", "ci-system", "Testing new model")
    transition_stage(REGISTRY_NAME, v2, "Production", "auto-pipeline", "Automated deploy")
except Exception as e:
    print("Gate blocked promotion:", e)


# Force simulate failure
client.transition_model_version_stage(REGISTRY_NAME, v2, "Production", archive_existing_versions=True)

log_audit_event(REGISTRY_NAME, v2, "Staging", "Production", "auto-pipeline", "BUG: metric gate bypassed")

# --- ROLLBACK

rollback_production(
    REGISTRY_NAME,
    v1,
    "ml-lead",
    "F1 dropped in production"
)
client.transition_model_version_stage(REGISTRY_NAME, v2, "Archived")
# --------------------------------------------------
# Final Governance Report
# --------------------------------------------------
report = GovernanceReport()
print("\n" + "="*60)
print("LoanGuard Governance Report")
print("="*60)
prod = report.current_production()
print(f"\nQ1 Current Production Model: {prod['name']} v{prod['version']}")
print(f"Q2 Promoted At: {prod['time']}")
print(f"Q3 Approved By: {prod['promoter']}")
run = client.get_run(client.get_model_version(REGISTRY_NAME, prod["version"]).run_id)
print(f"Q4 Training Dataset: {run.data.tags.get('dataset_version')}")
print("\nQ5 Metrics:")
for k,v in run.data.metrics.items():
    print(f"   {k}: {v:.4f}")
print("\nFull Changelog:")
for e in report.full_changelog(REGISTRY_NAME):
    print(f"{e['timestamp']} | v{e['version']} | {e['from_stage']} → {e['to_stage']} | {e['reason']}")

2026/03/12 17:28:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Simulating lifecycle...


2026/03/12 17:28:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LoanDefaultDetector' already exists. Creating a new version of this model...
2026/03/12 17:28:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 12df774f76f44fe2957799a4c96336d0 has no artifacts at artifact path 'model', registering model based on models:/m-2291bf05dfd647149c0776168985699c instead
Created version '4' of model 'LoanDefaultDetector'.
2026/03/12 17:28:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 17:28:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requ


LoanGuard Governance Report

Q1 Current Production Model: LoanDefaultDetector v4
Q2 Promoted At: 2026-03-12T17:28:43.515108
Q3 Approved By: ml-lead
Q4 Training Dataset: v1

Q5 Metrics:
   f1: 0.8706

Full Changelog:
2026-03-12T17:28:40.689265 | v4 | None → Staging | Initial validation passed
2026-03-12T17:28:40.711191 | v4 | Staging → Production | Approved for deployment
2026-03-12T17:28:43.448062 | v5 | None → Staging | Testing new model
2026-03-12T17:28:43.469838 | v5 | Staging → Production | Automated deploy
2026-03-12T17:28:43.484313 | v5 | Staging → Production | BUG: metric gate bypassed
2026-03-12T17:28:43.502532 | v5 | Production → Archived | ROLLBACK: F1 dropped in production
2026-03-12T17:28:43.515108 | v4 | Archived → Production | Rollback target promoted


Created version '5' of model 'LoanDefaultDetector'.


---
#  Lab Complete - Reflection
---

##  What You Built

| Task | What you implemented | Real-world equivalent |
|------|----------------------|----------------------|
| **Task 1** | DVC pointer files, drift detection, schema validator | DVC + Great Expectations |
| **Task 2** | MLflow experiment runs, multi-model comparison, artifact retrieval | MLflow Tracking Server |
| **Task 3** | Versioned model artifacts, semantic version logic, model registration | MLflow Registry + release process |
| **Task 4** | Stage transitions, audit log, rollback, governance queries | MLflow Registry Lifecycle + compliance |

---

## The Full Stack - How It Connects

```
loans_v1.csv  →  validate_schema()  →  [PASSES]  →  Training Run
                                                         │
                                              mlflow.log_params()
                                              mlflow.log_metrics()
                                              mlflow.log_model()
                                                         │
                                              register_model("LoanDefaultDetector")
                                                         │
                                         None → Staging → Production
                                                    (with audit log)
                                                         │
                             If it fails →  rollback_production()  →  Archived
```

---

##  Reflection Questions

Answer these in the cell below before the session ends:

1. Which task felt most like something you'd use immediately in your current project?
2. Where in LoanGuard's original problem could a schema validator have saved the most time?
3. If you had to pick ONE layer of the versioning stack to implement tomorrow - which would it be and why?

In [56]:
#  Your Reflection Answers
# ----------------------------------------------------

reflection = {
    "most_immediately_useful": """
    Task 2 (MLflow experiment tracking) felt the most immediately useful for my current work.
    It allows easy comparison of multiple models, tracking parameters, metrics, and artifacts in one place.
    This helps in understanding which model performs best and avoids confusion when running many experiments.
    """,

    "where_schema_validation_would_have_helped": """
    Schema validation would have helped during the data ingestion stage of LoanGuard's pipeline.
    If the dataset structure or column types changed (for example missing features or incorrect data types),
    the schema validator could have detected it early before training started.
    This would save time by preventing failed training runs and debugging later in the pipeline.
    """,

    "first_layer_to_implement": """
    I would implement experiment tracking first using MLflow.
    Tracking experiments, parameters, and metrics provides immediate visibility into model performance
    and ensures reproducibility. Once experiment tracking is in place, it becomes easier to add model
    versioning, registry, and governance layers later.
    """
}

for q, a in reflection.items():
    print(f"Q: {q}")
    print(f"A: {a.strip()}")
    print()

Q: most_immediately_useful
A: Task 2 (MLflow experiment tracking) felt the most immediately useful for my current work.
    It allows easy comparison of multiple models, tracking parameters, metrics, and artifacts in one place.
    This helps in understanding which model performs best and avoids confusion when running many experiments.

Q: where_schema_validation_would_have_helped
A: Schema validation would have helped during the data ingestion stage of LoanGuard's pipeline.
    If the dataset structure or column types changed (for example missing features or incorrect data types),
    the schema validator could have detected it early before training started.
    This would save time by preventing failed training runs and debugging later in the pipeline.

Q: first_layer_to_implement
A: I would implement experiment tracking first using MLflow.
    Tracking experiments, parameters, and metrics provides immediate visibility into model performance
    and ensures reproducibility. Once expe